# 01 — Tier 1 experiments

Prototype deterministic rules here, then move stable logic into `fraud_guard.tier1`.
Compare before/after with `fraud_guard.stats.filter_stats`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sqlalchemy import text

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]

from db.config import get_settings
get_settings.cache_clear()
from db.connection import get_engine

engine = get_engine()
VIEW = "dbo.TargetsByMetrics_RateGetAnswers"

df = pd.read_sql(f"SELECT * FROM {VIEW}", engine)
print("DB ok, shape:", df.shape)

DB ok, shape: (1212609, 35)


In [ ]:
##test-до фильтров

In [2]:
work = df[df["Question_ID"].eq(10012) & df["Answer_Value"].notna()].copy()
print("answered Q10012:", len(work))
print("top-box rate:", round((work["Answer_Value"] == 5).mean(), 4))

answered Q10012: 958598
top-box rate: 0.6809


In [4]:
##test-после фильтров(сотрудники/льготы)

##sum:Убрали ~2% строк (сотрудники/льготы).5% чуть снизился: 68.09% → 67.52% (−0.57 п.п.).


In [3]:
clean = work[work["BlackList"].eq("לא")].copy()
dropped = len(work) - len(clean)
print("kept (BlackList=לא):", len(clean))
print("dropped:", dropped, "share:", round(dropped / len(work), 4))
print("top-box before:", round((work["Answer_Value"] == 5).mean(), 4))
print("top-box after:", round((clean["Answer_Value"] == 5).mean(), 4))

kept (BlackList=לא): 938927
dropped: 19671 share: 0.0205
top-box before: 0.6809
top-box after: 0.6752


In [ ]:
## ключ клиента для поиска повторов.

In [5]:
def make_entity_key(row) -> str | None:
    for col in ("UserContact", "PhoneFromLog"):
        v = row[col]
        if pd.notna(v):
            s = str(v).strip()
            if s and s not in ("None", "nan"):
                return f"c:{s}"
    uid = row["ext_user_id"]
    if pd.notna(uid) and int(uid) != 0:
        return f"u:{int(uid)}"
    return None

clean["entity_key"] = clean.apply(make_entity_key, axis=1)
has_key = clean["entity_key"].notna()
print("with entity_key:", int(has_key.sum()), "share:", round(has_key.mean(), 4))
print("unique entity_key:", clean.loc[has_key, "entity_key"].nunique())
print("without entity_key:", int((~has_key).sum()))

with entity_key: 935018 share: 0.9958
unique entity_key: 284397
without entity_key: 3909


In [7]:
##  сколько раз отвечает один entity

#sum
#типичный клиент ответил 1 раз (медиана);
#но есть «тяжёлые» id: у 1% — 31+ ответов, максимум 5474;
#id с ≥10 ответами дают 446k строк (~почти половину clean) — резать всех подряд слишком грубо (там могут быть и лояльные клиенты за 1.5 года).

<function sum(iterable, /, start=0)>

In [6]:
vc = clean.loc[clean["entity_key"].notna()].groupby("entity_key").size()
print(vc.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("entities with >=3 answers:", int((vc >= 3).sum()))
print("entities with >=5 answers:", int((vc >= 5).sum()))
print("entities with >=10 answers:", int((vc >= 10).sum()))
print("answers from entities >=10:", int(vc[vc >= 10].sum()))

count    284397.000000
mean          3.287721
std          12.546833
min           1.000000
50%           1.000000
90%           7.000000
95%          13.000000
99%          31.000000
max        5474.000000
dtype: float64
entities with >=3 answers: 70316
entities with >=5 answers: 44990
entities with >=10 answers: 21344
answers from entities >=10: 446260


In [ ]:
##повторы в одном ресторане за один день

## sum
#Картина для «фермы на кассе» понятная:

#обычно 1 ответ на человека в ресторане за день;
#но есть пики до 30 за день;
#3268 групп с ≥3, 821 с ≥5 — это уже похоже на накрутку, не на «зашёл дважды».
#Возьмём порог ≥3 ответа / entity / store / day (можно потом сравнить с ≥5).

In [8]:
tmp = clean[clean["entity_key"].notna()].copy()
tmp["answer_day"] = pd.to_datetime(tmp["AnswerTime"]).dt.date

daily = (tmp.groupby(["entity_key", "PrintStore", "answer_day"])
           .size()
           .rename("n")
           .reset_index())

print(daily["n"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("groups with n>=2:", int((daily["n"] >= 2).sum()))
print("groups with n>=3:", int((daily["n"] >= 3).sum()))
print("groups with n>=5:", int((daily["n"] >= 5).sum()))

count    910660.000000
mean          1.026712
std           0.277646
min           1.000000
50%           1.000000
90%           1.000000
95%           1.000000
99%           2.000000
max          30.000000
Name: n, dtype: float64
groups with n>=2: 16790
groups with n>=3: 3268
groups with n>=5: 821


In [ ]:
## test

#Убрали
#14 072 строк (~1.5%)
#5% до
#67.52%
#5% после
67.13% (−0.39 п.п.)
#У удалённых
#93.5% пятёрок
#Простыми словами: повторы «один человек — один ресторан — один день ≥3 раза» почти все ставят 5. Это очень похоже на накрутку, а не на обычных клиентов.

In [10]:
tmp = clean[clean["entity_key"].notna()].copy()
tmp["answer_day"] = pd.to_datetime(tmp["AnswerTime"]).dt.date

gsize = tmp.groupby(["entity_key", "PrintStore", "answer_day"])["ParticipateNumber"].transform("size")
drop_ids = tmp.loc[gsize >= 3, "ParticipateNumber"]

after = clean[~clean["ParticipateNumber"].isin(drop_ids)].copy()

print("before (after BlackList):", len(clean))
print("dropped by freq>=3 same store-day:", len(clean) - len(after))
print("after:", len(after))
print("top-box before:", round((clean["Answer_Value"] == 5).mean(), 4))
print("top-box after:", round((after["Answer_Value"] == 5).mean(), 4))
print("top-box among dropped:", round((clean.loc[clean["ParticipateNumber"].isin(drop_ids), "Answer_Value"] == 5).mean(), 4))

before (after BlackList): 938927
dropped by freq>=3 same store-day: 14072
after: 924855
top-box before: 0.6752
top-box after: 0.6713
top-box among dropped: 0.935


In [ ]:
##эффект на рестораны (store×month)
# у скольких ресторан-месяцев score заметно изменился

#sum
#Простыми словами
#По сети эффект маленький (медиана delta ≈ 0).
#Но в отдельных ресторан-месяцах фильтр сильно меняет картину: у 60 ячеек −5 п.п. и больше, у 23 — −10 п.п. и больше.
#Худшие примеры: store 82 (2026-06/07) — с ~79–82% пятёрок до ~56–59% (−23 п.п.), объём ответов падает почти вдвое.
#То есть накрутка не «размазана» равномерно — она точечная по ресторанам/месяцам.

In [11]:
def five_pct(s):
    return 100.0 * (s == 5).mean()

before_panel = (clean.groupby(["PrintStore", "Year", "Month"])["Answer_Value"]
                  .agg(vol="count", five=five_pct))
after_panel = (after.groupby(["PrintStore", "Year", "Month"])["Answer_Value"]
                 .agg(vol="count", five=five_pct))

cmp = before_panel.join(after_panel, lsuffix="_before", rsuffix="_after", how="outer")
cmp["delta"] = cmp["five_after"] - cmp["five_before"]

print(cmp["delta"].describe(percentiles=[0.05, 0.5, 0.95]))
print("store-months with delta <= -5 pp:", int((cmp["delta"] <= -5).sum()))
print("store-months with delta <= -10 pp:", int((cmp["delta"] <= -10).sum()))
print("worst 10 deltas:")
print(cmp.nsmallest(10, "delta")[["vol_before", "five_before", "vol_after", "five_after", "delta"]])

count    4646.000000
mean       -0.273069
std         1.360178
min       -22.891232
5%         -1.577203
50%         0.000000
95%         0.074049
max         9.090909
Name: delta, dtype: float64
store-months with delta <= -5 pp: 60
store-months with delta <= -10 pp: 23
worst 10 deltas:
                       vol_before  five_before  vol_after  five_after  \
PrintStore Year Month                                                   
82.0       2026 7             212    78.773585        102   55.882353   
                6             264    81.818182        115   59.130435   
254.0      2026 7             305    71.147541        172   48.837209   
178.0      2026 6             231    80.519481        119   62.184874   
218.0      2026 7             294    79.931973        154   61.688312   
74.0       2026 7             229    84.279476        113   68.141593   
147.0      2026 6             178    79.213483        103   64.077670   
204.0      2026 6             340    76.764706        2

In [ ]:
##сводка «до → после» обоих правил

In [12]:
print("=== pipeline summary ===")
print("0 raw answered:", len(work), "top-box:", round((work["Answer_Value"]==5).mean(), 4))
print("1 after BlackList:", len(clean), "top-box:", round((clean["Answer_Value"]==5).mean(), 4))
print("2 after freq>=3:", len(after), "top-box:", round((after["Answer_Value"]==5).mean(), 4))
print("total dropped:", len(work) - len(after),
      "share:", round((len(work)-len(after))/len(work), 4))

=== pipeline summary ===
0 raw answered: 958598 top-box: 0.6809
1 after BlackList: 938927 top-box: 0.6752
2 after freq>=3: 924855 top-box: 0.6713
total dropped: 33743 share: 0.0352


In [ ]:
## один и тот же клиент почти всегда ставит 5 (при достаточном числе ответов).
#Важный нюанс: всегда 5 при ≥5 ответах — это 11 113 людей и ~149k ответов. Резать всех подряд рискованно: часть может быть просто очень довольными клиентами.

#Безопаснее для Tier 1: всегда пятёрки И много ответов, например top_rate == 1.0 и n >= 10.

In [13]:
ent = (after[after["entity_key"].notna()]
         .groupby("entity_key")["Answer_Value"]
         .agg(n="count", top_rate=lambda s: (s == 5).mean()))

# только у кого достаточно истории
ent5 = ent[ent["n"] >= 5]
print(ent5["top_rate"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("entities n>=5:", len(ent5))
print("entities n>=5 and top_rate==1.0:", int(((ent5["n"] >= 5) & (ent5["top_rate"] == 1.0)).sum()))
print("entities n>=5 and top_rate>=0.9:", int((ent5["top_rate"] >= 0.9).sum()))
print("answers from top_rate==1.0 (n>=5):", int(ent5.loc[ent5["top_rate"] == 1.0, "n"].sum()))

count    44854.000000
mean         0.625890
std          0.342658
min          0.000000
50%          0.714286
90%          1.000000
95%          1.000000
99%          1.000000
max          1.000000
Name: top_rate, dtype: float64
entities n>=5: 44854
entities n>=5 and top_rate==1.0: 11113
entities n>=5 and top_rate>=0.9: 13914
answers from top_rate==1.0 (n>=5): 148700


In [ ]:
## посмотреть объём при n≥10

#Объём уже осмысленнее: 4805 id, ~108k ответов (медиана 16 пятёрок подряд, максимум 314). Похоже на сильный кандидат в фильтр.

In [14]:
perf10 = ent[(ent["n"] >= 10) & (ent["top_rate"] == 1.0)]
print("entities always-5 with n>=10:", len(perf10))
print("answers always-5 with n>=10:", int(perf10["n"].sum()))
print(perf10["n"].describe())

entities always-5 with n>=10: 4805
answers always-5 with n>=10: 108005
count    4805.000000
mean       22.477627
std        20.718690
min        10.000000
25%        12.000000
50%        16.000000
75%        24.000000
max       314.000000
Name: n, dtype: float64


In [16]:
## применить правило

#sum
Правило сработало сильно.

#Убрали
#108 005 (~11.7%)
#Top-box
#67.13% → 62.78% (−4.35 п.п.)
#Простыми словами: «всегда пятёрка при ≥10 ответах» сильно снижает score. Это может быть и накрутка, и очень лояльные клиенты — спорное правило, его стоит отдельно согласовать со стейкхолдерами. Для эксперимента оно полезно как «верхняя оценка очистки».

In [15]:
bad_entities = set(perf10.index)

after2 = after[~(after["entity_key"].isin(bad_entities))].copy()
# строки без entity_key остаются

dropped = after["entity_key"].isin(bad_entities)
print("before:", len(after))
print("dropped always-5 n>=10:", int(dropped.sum()), "share:", round(dropped.mean(), 4))
print("after:", len(after2))
print("top-box before:", round((after["Answer_Value"] == 5).mean(), 4))
print("top-box after:", round((after2["Answer_Value"] == 5).mean(), 4))

before: 924855
dropped always-5 n>=10: 108005 share: 0.1168
after: 816850
top-box before: 0.6713
top-box after: 0.6278


In [ ]:
## полный pipeline summary

In [17]:
print("=== Tier1 pipeline ===")
print("0 raw answered:     ", len(work), " top-box:", round((work["Answer_Value"]==5).mean(), 4))
print("1 - BlackList:      ", len(clean), " top-box:", round((clean["Answer_Value"]==5).mean(), 4))
print("2 - freq store-day: ", len(after), " top-box:", round((after["Answer_Value"]==5).mean(), 4))
print("3 - always5 n>=10:  ", len(after2), " top-box:", round((after2["Answer_Value"]==5).mean(), 4))
print("total dropped:", len(work)-len(after2),
      "share:", round((len(work)-len(after2))/len(work), 4))
print("network 5% delta (pp):",
      round(100*((after2["Answer_Value"]==5).mean() - (work["Answer_Value"]==5).mean()), 2))

=== Tier1 pipeline ===
0 raw answered:      958598  top-box: 0.6809
1 - BlackList:       938927  top-box: 0.6752
2 - freq store-day:  924855  top-box: 0.6713
3 - always5 n>=10:   816850  top-box: 0.6278
total dropped: 141748 share: 0.1479
network 5% delta (pp): -5.31


In [18]:
## sum
#Итог простыми словами
#За три правила убрали ~15% ответов, сетевой 5% снизился с 68.1% до 62.8% (−5.3 п.п.).

#Правило	Надёжность	Эффект
#1. Убрать сотрудников/льготы (BlackList≠לא)
#Высокая
−0.6 п.п., мало строк
#2. ≥3 ответа один id в одном store за день
#Высокая
#−0.4 п.п.; у удалённых 93.5% пятёрок; сильный удар по отдельным ресторанам (−15…−23 п.п.)
#3. Всегда 5 при ≥10 ответах
#Средняя / спорная
#−4.4 п.п.; большой объём — нужно согласование